In [2]:
Interesting_regions =   [4, 5, 32, 33, 44, 45, 48, 49]
Interesting_regions_right =   [5, 33, 45, 49]
Interesting_regions_left = [4, 32, 44, 48,]

Good_subjects =         [0, 2, 5, 9, 12, 14]
Ok_subjects =           [1, 3, 4, 6, 11, 13, 19]
Other_subjects = [i for i in range(20) if i not in Ok_subjects and i not in Good_subjects]

Subjects_dict = {"Good_subjects": Good_subjects, "Ok_subjects":Ok_subjects, "Other_subjects":Other_subjects}

data_folder="../Data/"


from BCI_Library import read_subject, plot_brain, train_models, extract_features_more_bands, saveresults_pickle, select_rows
from BCI_Library import sort_rank, compute_welch_select_regions, extract_features_from_spectra_more_bands,compute_welch
from BCI_Library import plot_hist2d_selected_regions_folds, train_single_model, select_regions_MyCriteria, select_regions_Cohen_effect_size
from BCI_Library import global_selection, build_compiled_mlp

"""# new_row = {
#     "ROIs": [selected_regions_EEG],
#     "NROIs":[25]
#     "DataType": ["EEG"],
#     "freqs_band": [((4,8),(8,12),(12,30))],  
#     "subject": [subject_index],
#     "Classifier": ["LDA"],
#     "Accuracy": [training[0]],
#     "Features": ["PowerSpectra - MeanMax band"],
#     "Comments": ["-"],
#     "Region Selection": ["-"]
#     "Date": [today],
#     "Personalized_Freq_Range" = False,
# }"""


2026-01-29 11:00:28.640722: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-29 11:00:28.717415: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-29 11:00:30.264518: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


'# new_row = {\n#     "ROIs": [selected_regions_EEG],\n#     "NROIs":[25]\n#     "DataType": ["EEG"],\n#     "freqs_band": [((4,8),(8,12),(12,30))],  \n#     "subject": [subject_index],\n#     "Classifier": ["LDA"],\n#     "Accuracy": [training[0]],\n#     "Features": ["PowerSpectra - MeanMax band"],\n#     "Comments": ["-"],\n#     "Region Selection": ["-"]\n#     "Date": [today],\n#     "Personalized_Freq_Range" = False,\n# }'

# Trainings from 15/12/2025 


Selection of regions in Kfolds, different selections criteria, consider only alpha beta bands

## stratified k fold - 5 folds

### Cohen's d effect size

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import date
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import StratifiedKFold
import tqdm

seed = 224

saveresults = True
filepath = "../Results/BCI_Performances/BCI_Performances_region_selection_on_folds_8_30_Hz.pkl"
file_input = filepath
file_output = filepath
miff = [8,12]
maff = [12,30]
freqs_band = tuple([(i,j) for i,j in zip(miff,maff)])
Features_name = "PowerSpectra - MeanMax band"
Region_Selection= "Cohen's d effect size selection" # "-"
Comments = "MLP hidden neurons [20], epochs 20, batch size 32"
PFR = False



n_folds=5
kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
today = date.today()
sfreq = 250

tutte_roi = 68
verbose = False

Classifier_names, meths = ["MLP"], ["mlp"] # ["LDA", "SVC", "RandomForest"],[LDA, SVC, RandomForestClassifier]
MLP_fit_kwargs={'epochs':20, 'batch_size':32}

for DataType in ["EEG","MEG"]:
    for Classifier_name, meth in zip(Classifier_names, meths):

        Rows = pd.DataFrame({
            "ROIs": [],
            "NROIs":[],
            "DataType": [],
            "freqs_band": [],  
            "subject": [],
            "Classifier": [],
            "Accuracy": [],
            "Features": [],
            "Comments": [],
            "Region Selection": [],
            "Date": []
        })
       
        print("\n\n"+"#"*30+"\n",DataType,Classifier_name)
        for subject_index in tqdm.tqdm(range(20)):

            data_2_MI = read_subject(data_folder, DataType, "MI", subject_index, verbose=False)
            data_2_Rest = read_subject(data_folder, DataType, "Baseline", subject_index, verbose=False)

            WMI, WRe, FreqMI = compute_welch(data_2_MI, data_2_Rest, sfreq,
                                                            kargs_welch={"fmin":miff[0], "fmax":maff[-1]})
            # WMI.shape (trials, ROIs, freqs)

            labels = [1 for __ in range(len(WMI))] + [0 for __ in range(len(WRe))] 

            Feat_extraction = extract_features_from_spectra_more_bands(WMI, WRe, [ _ for _ in range(tutte_roi)], FreqMI,
                    min_freq_frature=miff, max_freq_frature=maff, 
                    select_mean=True, select_max=True,
                    reshape_features=False)


            Feat_MI, Feat_Rest, labls_MI, labls_Rest, all_regions = Feat_extraction
            #print("\n\n"+"#"*30+"featureMI.shape",Feat_MI.shape)
            # Feat_MI.shape (Trials, selected_ROIs, features for Roi)

            Feat_All = np.concatenate((Feat_MI, Feat_Rest), axis=0)
            #print("\n\n"+"#"*30+"Feat_All.shape",Feat_All.shape)
            labels_All = np.array(labels)

            for num_best_ROIs in [3,5,10,20]:#40

                fold_accuracies = []

                for fold_idx, (train_idx, val_idx) in enumerate(kf.split([0 for _ in range(len(labels))], labels)):

                    # selecting region based on training data only
                    training_index_MI = train_idx[np.where(train_idx<len(WMI))]
                    training_index_Re = train_idx[np.where(train_idx>=len(WMI))]-len(WMI)

                    regions_selected, effect_size  = select_regions_Cohen_effect_size(wpsdMI=WMI[training_index_MI], 
                                                                    wpsdRest=WRe[training_index_Re], 
                                                                    important_regions=[], nbest_regions=num_best_ROIs)

                    
                    Feat_All_regions_selected = Feat_All[:,regions_selected,:].reshape(Feat_All.shape[0],-1)
                    #print("\n\n"+"#"*30+"Feat_All_regions_selected.shape",Feat_All_regions_selected.shape)
                    #####################################################################################################################

                    if Classifier_name.lower()=="mlp":
                        meth = build_compiled_mlp(input_dim=Feat_All_regions_selected.shape[1])
                        
                    acc, cm, model = train_single_model(Feat_All_regions_selected, labels_All,
                                                        train_idx, val_idx, method=meth, seed=224, fit_kwargs=MLP_fit_kwargs)

                    fold_accuracies.append(acc)
                    if verbose:
                        print(f"Fold {fold_idx+1} accuracy: {acc:.4f}, Confusion matrix:")
                        print(cm)

                    #####################################################################################################################

                if verbose:
                    print("\n==================================\n","Mean accuracy:", np.mean(fold_accuracies))
                    print("Std accuracy:", np.std(fold_accuracies))

                new_row = {
                    "ROIs": [regions_selected],
                    "NROIs":[len(regions_selected)],
                    "DataType": [DataType],
                    "freqs_band": [freqs_band],  
                    "subject": [subject_index],
                    "Classifier": [Classifier_name], #SVC, #RandomForest, #LDA
                    "Accuracy": [fold_accuracies],
                    "Features": [Features_name],
                    "Comments": [Comments],
                    "Region Selection": [Region_Selection],
                    "Date": [today],
                    "Personalized_Freq_Range": [PFR],
                }
                Rows=pd.concat((Rows,pd.DataFrame(new_row)))
        

        if saveresults:
            saveresults_pickle(Rows, inputfile=file_input, outputfile=file_output, backup=True)



##############################
 EEG MLP


  0%|          | 0/20 [00:00<?, ?it/s]

I0000 00:00:1769680838.014250 1011321 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6821 MB memory:  -> device: 0, name: Quadro RTX 4000, pci bus id: 0000:5b:00.0, compute capability: 7.5
I0000 00:00:1769680838.015210 1011321 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 6761 MB memory:  -> device: 1, name: Quadro RTX 4000, pci bus id: 0000:9e:00.0, compute capability: 7.5


Epoch 1/20


2026-01-29 11:00:40.537946: I external/local_xla/xla/service/service.cc:163] XLA service 0x7f64b4008c40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-01-29 11:00:40.537983: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): Quadro RTX 4000, Compute Capability 7.5
2026-01-29 11:00:40.537992: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (1): Quadro RTX 4000, Compute Capability 7.5
2026-01-29 11:00:40.587457: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-01-29 11:00:40.757201: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91800


...

Epoch 1/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 409ms/step - accuracy: 0.3741 - loss: 0.9754 - val_accuracy: 0.4595 - val_loss: 0.9312
Epoch 2/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.3810 - loss: 0.9171 - val_accuracy: 0.4595 - val_loss: 0.8771
Epoch 3/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.3810 - loss: 0.8724 - val_accuracy: 0.4595 - val_loss: 0.8266
Epoch 4/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.4286 - loss: 0.8282 - val_accuracy: 0.4865 - val_loss: 0.7834
Epoch 5/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.4218 - loss: 0.7908 - val_accuracy: 0.5135 - val_loss: 0.7464
Epoch 6/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.4286 - loss: 0.7604 - val_accuracy: 0.4865 - val_loss: 0.7142
Epoch 7/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.4490 - loss: 0.7318 - val_accuracy: 0.4595 - val_loss: 0.6880
Epoch 8/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.4626 - loss: 0.7101 - val_accuracy: 0.4324 - val_loss: 0.6654

 50%|█████     | 10/20 [27:34<27:48, 166.83s/it]

Epoch 1/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 401ms/step - accuracy: 0.2745 - loss: 0.9960 - val_accuracy: 0.3333 - val_loss: 1.0047
Epoch 2/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.2876 - loss: 0.9448 - val_accuracy: 0.3333 - val_loss: 0.9608
Epoch 3/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.3203 - loss: 0.8977 - val_accuracy: 0.3590 - val_loss: 0.9200
Epoch 4/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.3725 - loss: 0.8559 - val_accuracy: 0.3590 - val_loss: 0.8818
Epoch 5/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.3922 - loss: 0.8138 - val_accuracy: 0.3846 - val_loss: 0.8471
Epoch 6/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.4248 - loss: 0.7787 - val_accuracy: 0.3846 - val_loss: 0.8150
Epoch 7/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.4575 - loss: 0.7449 - val_accuracy: 0.3846 - val_loss: 0.7854
Epoch 8/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.4706 - loss: 0.7129 - val_accuracy: 0.5128 - val_loss: 0.7582

 55%|█████▌    | 11/20 [30:23<25:05, 167.29s/it]

Epoch 1/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 395ms/step - accuracy: 0.4183 - loss: 0.8158 - val_accuracy: 0.4615 - val_loss: 0.7673
Epoch 2/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.4248 - loss: 0.7744 - val_accuracy: 0.4872 - val_loss: 0.7323
Epoch 3/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.4837 - loss: 0.7393 - val_accuracy: 0.4872 - val_loss: 0.7004
Epoch 4/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.5294 - loss: 0.7070 - val_accuracy: 0.5897 - val_loss: 0.6726
Epoch 5/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.5817 - loss: 0.6807 - val_accuracy: 0.6667 - val_loss: 0.6480
Epoch 6/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.6078 - loss: 0.6537 - val_accuracy: 0.6667 - val_loss: 0.6278
Epoch 7/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.6144 - loss: 0.6350 - val_accuracy: 0.6923 - val_loss: 0.6100
Epoch 8/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.6275 - loss: 0.6164 - val_accuracy: 0.7179 - val_loss: 0.5946

In [ ]:
#Integrate EEG EMG
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import date
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import StratifiedKFold
import tqdm

seed = 224

# keys of the performance dictionary
saveresults = True
filepath = "../Results/BCI_Performances/BCI_Performances_region_selection_on_folds_8_30_Hz.pkl"
file_input = filepath
file_output = filepath
miff = [8,12]
maff = [12,30]
freqs_band = tuple([(i,j) for i,j in zip(miff,maff)])
Features_name = "PowerSpectra - MeanMax band"
Region_Selection= "Cohen's d effect size selection" # "-"
Comments = "MLP hidden neurons [20], epochs 20, batch size 32"
PFR = False

n_folds=5
kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
today = date.today()
sfreq = 250

tutte_roi = 68
verbose = False

Classifier_names, meths = ["MLP"], ["mlp"] # ["LDA", "SVC", "RandomForest"],[LDA, SVC, RandomForestClassifier]
MLP_fit_kwargs={'epochs':20, 'batch_size':32}

for DataType in ["EEG concat MEG"]:
    for Classifier_name, meth in zip(Classifier_names, meths):

        Rows = pd.DataFrame({
            "ROIs": [],
            "NROIs":[],
            "DataType": [],
            "freqs_band": [],  
            "subject": [],
            "Classifier": [],
            "Accuracy": [],
            "Features": [],
            "Comments": [],
            "Region Selection": [],
            "Date": []
        })
        print("\n\n"+"#"*30+"\n",DataType,Classifier_name)
        for subject_index in tqdm.tqdm(range(20)):

            data_2_MI_EEG = read_subject(data_folder, "EEG", "MI",subject_index, verbose=False)
            data_2_Rest_EEG =read_subject(data_folder, "EEG", "Baseline",subject_index, verbose=False)
            data_2_MI_MEG = read_subject(data_folder, "MEG", "MI",subject_index, verbose=False)
            data_2_Rest_MEG =read_subject(data_folder, "MEG", "Baseline",subject_index, verbose=False)

            WMI_EEG, WRe_EEG, FreqMI_EEG = compute_welch(data_2_MI_EEG, data_2_Rest_EEG, sfreq,
                                                            kargs_welch={"fmin":miff[0], "fmax":maff[-1]})
            # WMI.shape (trials, ROIs, freqs)
            labels_EEG = [1 for __ in range(len(WMI_EEG))] + [0 for __ in range(len(WRe_EEG))] 
            Feat_extraction_EEG = extract_features_from_spectra_more_bands(WMI_EEG, WRe_EEG, [ _ for _ in range(tutte_roi)], FreqMI_EEG,
                    min_freq_frature=miff, max_freq_frature=maff, 
                    select_mean=True, select_max=True,
                    reshape_features=False)

            Feat_MI_EEG, Feat_Rest_EEG, labls_MI_EEG, labls_Rest_EEG, all_regions_EEG = Feat_extraction_EEG
            # Feat_MI.shape (Trials, selected_ROIs, features for Roi)
            Feat_All_EEG = np.concatenate((Feat_MI_EEG, Feat_Rest_EEG), axis=0)
            labels_All_EEG= labels_EEG

            #################################################################################

            WMI_MEG, WRe_MEG, FreqMI_MEG = compute_welch(data_2_MI_MEG, data_2_Rest_MEG, sfreq,
                                                            kargs_welch={"fmin":miff[0], "fmax":maff[-1]})
            labels_MEG = [1 for __ in range(len(WMI_MEG))] + [0 for __ in range(len(WRe_MEG))] 
            Feat_extraction_MEG = extract_features_from_spectra_more_bands(WMI_MEG, WRe_MEG, [ _ for _ in range(tutte_roi)], FreqMI_MEG,
                    min_freq_frature=miff, max_freq_frature=maff, 
                    select_mean=True, select_max=True,
                    reshape_features=False)
            Feat_MI_MEG, Feat_Rest_MEG, labls_MI_MEG, labls_Rest_MEG, all_regions_MEG = Feat_extraction_MEG
            Feat_All_MEG = np.concatenate((Feat_MI_MEG, Feat_Rest_MEG), axis=0)
            labels_All_MEG= labels_MEG


            for num_best_ROIs in [3,5,10,20]: #40

                fold_accuracies = []

                for fold_idx, (train_idx, val_idx) in enumerate(kf.split([0 for _ in range(len(labels_MEG))], labels_MEG)):

                    # selecting region based on training data only
                    training_index_MI = train_idx[np.where(train_idx<len(WMI_EEG))]
                    training_index_Re = train_idx[np.where(train_idx>=len(WMI_EEG))]-len(WMI_EEG)

                    regions_selected_EEG, effect_size = select_regions_Cohen_effect_size(wpsdMI=WMI_EEG[training_index_MI], 
                                                                    wpsdRest=WRe_EEG[training_index_MI], 
                                                                    important_regions=[], nbest_regions=num_best_ROIs)
                    
                    regions_selected_MEG, effect_size = select_regions_Cohen_effect_size(wpsdMI=WMI_MEG[training_index_MI], 
                                                                    wpsdRest=WRe_MEG[training_index_MI], 
                                                                    important_regions=[], nbest_regions=num_best_ROIs)

                    
                    Feat_All_regions_selected_EEG = Feat_All_EEG[:,regions_selected_EEG,:].reshape(Feat_All_EEG.shape[0],-1)
                    Feat_All_regions_selected_MEG = Feat_All_MEG[:,regions_selected_MEG,:].reshape(Feat_All_MEG.shape[0],-1)
                    # Feat_All_regions_selected_EEG.shape is (trials,features)
                    Feat_All_regions_selected_concat = np.concatenate((Feat_All_regions_selected_EEG,Feat_All_regions_selected_MEG),axis=-1)
                    #####################################################################################################################
                    acc, cm, model = train_single_model(Feat_All_regions_selected_concat, np.array(labels_All_MEG),
                                                        train_idx, val_idx, method=meth, seed=224)

                    if Classifier_name.lower()=="mlp":
                        meth = build_compiled_mlp(input_dim=Feat_All_regions_selected_concat.shape[1])
                        
                    acc, cm, model = train_single_model(Feat_All_regions_selected_concat, labels_All_MEG,
                                                        train_idx, val_idx, method=meth, seed=224, fit_kwargs=MLP_fit_kwargs)

                    fold_accuracies.append(acc)
                    if verbose:
                        print(f"Fold {fold_idx+1} accuracy: {acc:.4f}, Confusion matrix:")
                        print(cm)

                    #####################################################################################################################

                # Summary
                if verbose:
                    print("\n==================================")
                    print("Mean accuracy:", np.mean(fold_accuracies))
                    print("Std accuracy:", np.std(fold_accuracies))



                new_row = {
                    "ROIs": [[regions_selected_EEG,regions_selected_MEG]],
                    "NROIs":[len(regions_selected_EEG)],
                    "DataType": [DataType],
                    "freqs_band": [freqs_band],  
                    "subject": [subject_index],
                    "Classifier": [Classifier_name], #SVC, #RandomForest, #LDA
                    "Accuracy": [fold_accuracies],
                    "Features": [Features_name],
                    "Comments": [Comments], 
                    "Region Selection": [Region_Selection],
                    "Date": [today],
                    "Personalized_Freq_Range": [PFR],
                }
                Rows=pd.concat((Rows,pd.DataFrame(new_row)))
        


        if saveresults:
            saveresults_pickle(Rows, inputfile=file_input, outputfile=file_output, backup=True)



##############################
 EEG concat MEG SVC


100%|██████████| 20/20 [00:50<00:00,  2.51s/it]


There are 0 duplicated items (based on all columns except Date, ROIs, Performance).
 Index of the duplicated items: []


##############################
 EEG concat MEG RandomForest


100%|██████████| 20/20 [01:14<00:00,  3.75s/it]


There are 0 duplicated items (based on all columns except Date, ROIs, Performance).
 Index of the duplicated items: []


##############################
 EEG concat MEG LDA


100%|██████████| 20/20 [00:52<00:00,  2.60s/it]

There are 0 duplicated items (based on all columns except Date, ROIs, Performance).
 Index of the duplicated items: []


### Global Region Selection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import date
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import StratifiedKFold
import tqdm

today = date.today()
sfreq = 250
seed = 224

global_selection_base = "MyCriteria" #"Cohen", "MyCriteria"
selection_method = "MyCriteria" if global_selection_base=="MyCriteria" else "Cohen_effect_size" if global_selection_base=="Cohen" else None


saveresults = True
filepath = "../Results/BCI_Performances/BCI_Performances_region_selection_on_folds_8_30_Hz.pkl"
file_input = filepath
file_output = filepath
miff = [8,12]
maff = [12,30]
freqs_band = tuple([(i,j) for i,j in zip(miff,maff)])
Features_name = "PowerSpectra - MeanMax band"
Region_Selection= f"Global Selection - {global_selection_base} based" # "-"
Comments = "-"
PFR = False


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

tutte_roi = 68
verbose = False

for DataType in ["EEG","MEG"]:
    for Classifier_name, meth in zip(["SVC", "RandomForest", "LDA"],[SVC, RandomForestClassifier, LDA]):

        Rows = pd.DataFrame({
            "ROIs": [],
            "NROIs":[],
            "DataType": [],
            "freqs_band": [],  
            "subject": [],
            "Classifier": [],
            "Accuracy": [],
            "Features": [],
            "Comments": [],
            "Region Selection": [],
            "Date": []
        })
       
        print("\n\n"+"#"*30+"\n",DataType,Classifier_name, global_selection_base)

        regions_selected  = global_selection(selection_method, DataType)
        print(regions_selected)

        for subject_index in tqdm.tqdm(range(20)):

            data_2_MI = read_subject(data_folder, DataType, "MI", subject_index, verbose=False)
            data_2_Rest = read_subject(data_folder, DataType, "Baseline", subject_index, verbose=False)

            WMI, WRe, FreqMI = compute_welch(data_2_MI, data_2_Rest, sfreq,
                                                            kargs_welch={"fmin":miff[0], "fmax":maff[-1]})
            # WMI.shape (trials, ROIs, freqs)

            labels = [1 for __ in range(len(WMI))] + [0 for __ in range(len(WRe))] 

            Feat_extraction = extract_features_from_spectra_more_bands(WMI, WRe, [ _ for _ in range(tutte_roi)], FreqMI,
                    min_freq_frature=miff, max_freq_frature=maff, 
                    select_mean=True, select_max=True,
                    reshape_features=False)


            Feat_MI, Feat_Rest, labls_MI, labls_Rest, all_regions = Feat_extraction
            #print("\n\n"+"#"*30+"featureMI.shape",Feat_MI.shape)
            # Feat_MI.shape (Trials, selected_ROIs, features for Roi)

            Feat_All = np.concatenate((Feat_MI, Feat_Rest), axis=0)
            #print("\n\n"+"#"*30+"Feat_All.shape",Feat_All.shape)
            labels_All = np.array(labels)


            fold_accuracies = []

            for fold_idx, (train_idx, val_idx) in enumerate(skf.split([0 for _ in range(len(labels))], labels)):


                Feat_All_regions_selected = Feat_All[:,regions_selected,:].reshape(Feat_All.shape[0],-1)
                #print("\n\n"+"#"*30+"Feat_All_regions_selected.shape",Feat_All_regions_selected.shape)
                #####################################################################################################################

                acc, cm, model = train_single_model(Feat_All_regions_selected, labels_All,
                                                    train_idx, val_idx, method=meth, seed=224)

                fold_accuracies.append(acc)
                if verbose:
                    print(f"Fold {fold_idx+1} accuracy: {acc:.4f}, Confusion matrix:")
                    print(cm)

                #####################################################################################################################

            if verbose:
                print("\n==================================\n","Mean accuracy:", np.mean(fold_accuracies))
                print("Std accuracy:", np.std(fold_accuracies))

            new_row = {
                "ROIs": [regions_selected],
                "NROIs":[len(regions_selected)],
                "DataType": [DataType],
                "freqs_band": [freqs_band],  
                "subject": [subject_index],
                "Classifier": [Classifier_name], #SVC, #RandomForest, #LDA
                "Accuracy": [fold_accuracies],
                "Features": [Features_name],
                "Comments": [Comments],
                "Region Selection": [Region_Selection],
                "Date": [today],
                "Personalized_Freq_Range": [PFR],
            }
            Rows=pd.concat((Rows,pd.DataFrame(new_row)))
    

        if saveresults:
            saveresults_pickle(Rows, inputfile=file_input, outputfile=file_output, backup=True)

In [ ]:
#Integrate EEG EMG
import mne
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import date
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import StratifiedKFold
import tqdm

seed = 224

for global_selection_base in ["MyCriteria", "Cohen"]:
    selection_method = "MyCriteria" if global_selection_base=="MyCriteria" else "Cohen_effect_size" if global_selection_base=="Cohen" else None

    saveresults = True
    filepath = "../Results/BCI_Performances/BCI_Performances_region_selection_on_folds_8_30_Hz.pkl"
    file_input = filepath
    file_output = filepath
    miff = [8,12]
    maff = [12,30]
    freqs_band = tuple([(i,j) for i,j in zip(miff,maff)])
    Features_name = "PowerSpectra - MeanMax band"
    Region_Selection= f"Global Selection - {global_selection_base} based" # "-"
    Comments = "-"
    PFR = False

    n_folds=5
    skf = StratifiedKFold(n_splits=5, n_repeats=2, random_state=seed)
    today = date.today()
    sfreq = 250

    tutte_roi = 68
    verbose = False


    regions_selected_EEG  = global_selection(selection_method, "EEG")
    regions_selected_MEG =  global_selection(selection_method, "MEG")
    print("selected EEG", regions_selected_EEG) 
    print("selected MEG", regions_selected_MEG) 


    for DataType in ["EEG concat MEG"]:
        for Classifier_name, meth in zip(["SVC", "RandomForest", "LDA"],[SVC, RandomForestClassifier, LDA]):

            Rows = pd.DataFrame({
                "ROIs": [],
                "NROIs":[],
                "DataType": [],
                "freqs_band": [],  
                "subject": [],
                "Classifier": [],
                "Accuracy": [],
                "Features": [],
                "Comments": [],
                "Region Selection": [],
                "Date": []
            })
            print("\n\n"+"#"*30+"\n",global_selection_base,DataType,Classifier_name)
            for subject_index in tqdm.tqdm(range(20)):

                data_2_MI_EEG = read_subject(data_folder, "EEG", "MI",subject_index, verbose=False)
                data_2_Rest_EEG =read_subject(data_folder, "EEG", "Baseline",subject_index, verbose=False)
                data_2_MI_MEG = read_subject(data_folder, "MEG", "MI",subject_index, verbose=False)
                data_2_Rest_MEG =read_subject(data_folder, "MEG", "Baseline",subject_index, verbose=False)

                WMI_EEG, WRe_EEG, FreqMI_EEG = compute_welch(data_2_MI_EEG, data_2_Rest_EEG, sfreq,
                                                                kargs_welch={"fmin":miff[0], "fmax":maff[-1]})
                # WMI.shape (trials, ROIs, freqs)
                labels_EEG = [1 for __ in range(len(WMI_EEG))] + [0 for __ in range(len(WRe_EEG))] 
                Feat_extraction_EEG = extract_features_from_spectra_more_bands(WMI_EEG, WRe_EEG, [ _ for _ in range(tutte_roi)], FreqMI_EEG,
                        min_freq_frature=miff, max_freq_frature=maff, 
                        select_mean=True, select_max=True,
                        reshape_features=False)

                Feat_MI_EEG, Feat_Rest_EEG, labls_MI_EEG, labls_Rest_EEG, all_regions_EEG = Feat_extraction_EEG
                # Feat_MI.shape (Trials, selected_ROIs, features for Roi)
                Feat_All_EEG = np.concatenate((Feat_MI_EEG, Feat_Rest_EEG), axis=0)
                labels_All_EEG= labels_EEG

                #################################################################################

                WMI_MEG, WRe_MEG, FreqMI_MEG = compute_welch(data_2_MI_MEG, data_2_Rest_MEG, sfreq,
                                                                kargs_welch={"fmin":miff[0], "fmax":maff[-1]})
                labels_MEG = [1 for __ in range(len(WMI_MEG))] + [0 for __ in range(len(WRe_MEG))] 
                Feat_extraction_MEG = extract_features_from_spectra_more_bands(WMI_MEG, WRe_MEG, [ _ for _ in range(tutte_roi)], FreqMI_MEG,
                        min_freq_frature=miff, max_freq_frature=maff, 
                        select_mean=True, select_max=True,
                        reshape_features=False)
                Feat_MI_MEG, Feat_Rest_MEG, labls_MI_MEG, labls_Rest_MEG, all_regions_MEG = Feat_extraction_MEG
                Feat_All_MEG = np.concatenate((Feat_MI_MEG, Feat_Rest_MEG), axis=0)
                labels_All_MEG= labels_MEG




                fold_accuracies = []

                for fold_idx, (train_idx, val_idx) in enumerate(skf.split([0 for _ in range(len(labels_MEG))], labels_MEG)):
                    

                    Feat_All_regions_selected_EEG = Feat_All_EEG[:,regions_selected_EEG,:].reshape(Feat_All_EEG.shape[0],-1)
                    Feat_All_regions_selected_MEG = Feat_All_MEG[:,regions_selected_MEG,:].reshape(Feat_All_MEG.shape[0],-1)
                    # Feat_All_regions_selected_EEG.shape is (trials,features)
                    Feat_All_regions_selected_concat = np.concatenate((Feat_All_regions_selected_EEG,Feat_All_regions_selected_MEG),axis=-1)
                    #####################################################################################################################
                    acc, cm, model = train_single_model(Feat_All_regions_selected_concat, np.array(labels_All_MEG),
                                                        train_idx, val_idx, method=meth, seed=224)

                    fold_accuracies.append(acc)
                    if verbose:
                        print(f"Fold {fold_idx+1} accuracy: {acc:.4f}, Confusion matrix:")
                        print(cm)

                    #####################################################################################################################

                # Summary
                if verbose:
                    print("\n==================================")
                    print("Mean accuracy:", np.mean(fold_accuracies))
                    print("Std accuracy:", np.std(fold_accuracies))



                new_row = {
                    "ROIs": [[regions_selected_EEG,regions_selected_MEG]],
                    "NROIs":[len(regions_selected_EEG)],
                    "DataType": [DataType],
                    "freqs_band": [freqs_band],  
                    "subject": [subject_index],
                    "Classifier": [Classifier_name], #SVC, #RandomForest, #LDA
                    "Accuracy": [fold_accuracies],
                    "Features": [Features_name],
                    "Comments": [Comments],
                    "Region Selection": [Region_Selection],
                    "Date": [today],
                    "Personalized_Freq_Range": [PFR],
                }
                Rows=pd.concat((Rows,pd.DataFrame(new_row)))
            


            if saveresults:
                saveresults_pickle(Rows, inputfile=file_input, outputfile=file_output, backup=True)

# On 1 region

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import date
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import StratifiedKFold
import tqdm

today = date.today()
sfreq = 250

seed = 224

saveresults = True
filepath = "../Results/BCI_Performances/BCI_Performances_on_single_region.pkl"
file_input = filepath
file_output = filepath
miff = [8,12]
maff = [12,30]
freqs_band = tuple([(i,j) for i,j in zip(miff,maff)])
Features_name = "PowerSpectra - MeanMax band"
Region_Selection= "-" # "-"
Comments = "-"
PFR = False

n_folds=5
kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
today = date.today()
sfreq = 250

tutte_roi = 68
verbose = False

Classifier_names, meths = ["LDA", "SVC", "RandomForest"],[LDA, SVC, RandomForestClassifier] # ["MLP"], ["mlp"] # CI VUOLE TROPPO TEMPO PER MLP
MLP_fit_kwargs={'epochs':20, 'batch_size':32}

for DataType in ["EEG","MEG"]:
    for Classifier_name, meth in zip(Classifier_names, meths):

        Rows = pd.DataFrame({
            "ROIs": [],
            "NROIs":[],
            "DataType": [],
            "freqs_band": [],  
            "subject": [],
            "Classifier": [],
            "Accuracy": [],
            "Features": [],
            "Comments": [],
            "Region Selection": [],
            "Date": []
        })
       
        print("\n\n"+"#"*30+"\n",DataType,Classifier_name)
        for subject_index in tqdm.tqdm(range(20)):

            data_2_MI = read_subject(data_folder, DataType, "MI", subject_index, verbose=False)
            data_2_Rest = read_subject(data_folder, DataType, "Baseline", subject_index, verbose=False)

            WMI, WRe, FreqMI = compute_welch(data_2_MI, data_2_Rest, sfreq,
                                                            kargs_welch={"fmin":miff[0], "fmax":maff[-1]})
            # WMI.shape (trials, ROIs, freqs)

            labels = [1 for __ in range(len(WMI))] + [0 for __ in range(len(WRe))] 

            Feat_extraction = extract_features_from_spectra_more_bands(WMI, WRe, [ _ for _ in range(tutte_roi)], FreqMI,
                    min_freq_frature=miff, max_freq_frature=maff, 
                    select_mean=True, select_max=True,
                    reshape_features=False)


            Feat_MI, Feat_Rest, labls_MI, labls_Rest, all_regions = Feat_extraction
            #print("\n\n"+"#"*30+"featureMI.shape",Feat_MI.shape)
            # Feat_MI.shape (Trials, selected_ROIs, features for Roi)

            Feat_All = np.concatenate((Feat_MI, Feat_Rest), axis=0)
            #print("\n\n"+"#"*30+"Feat_All.shape",Feat_All.shape)
            labels_All = np.array(labels)


            for region_idx in range(tutte_roi):

                regions_selected = [region_idx]
                fold_accuracies = []

                for fold_idx, (train_idx, val_idx) in enumerate(kf.split([0 for _ in range(len(labels))], labels)):
                    
                    Feat_All_regions_selected = Feat_All[:,regions_selected,:].reshape(Feat_All.shape[0],-1)
                    #print("\n\n"+"#"*30+"Feat_All_regions_selected.shape",Feat_All_regions_selected.shape)
                    #####################################################################################################################

                    if Classifier_name.lower()=="mlp":
                        meth = build_compiled_mlp(input_dim=Feat_All_regions_selected.shape[1])
                        
                    acc, cm, model = train_single_model(Feat_All_regions_selected, labels_All,
                                                        train_idx, val_idx, method=meth, seed=224, fit_kwargs=MLP_fit_kwargs)

                    fold_accuracies.append(acc)
                    if verbose:
                        print(f"Fold {fold_idx+1} accuracy: {acc:.4f}, Confusion matrix:")
                        print(cm)

                    #####################################################################################################################

                if verbose:
                    print("\n==================================\n","Mean accuracy:", np.mean(fold_accuracies))
                    print("Std accuracy:", np.std(fold_accuracies))

                new_row = {
                    "ROIs": [regions_selected],
                    "NROIs":[len(regions_selected)],
                    "DataType": [DataType],
                    "freqs_band": [freqs_band],  
                    "subject": [subject_index],
                    "Classifier": [Classifier_name], #SVC, #RandomForest, #LDA, #MLP
                    "Accuracy": [fold_accuracies],
                    "Features": [Features_name],
                    "Comments": [Comments],
                    "Date": [today],
                    "Personalized_Freq_Range": [PFR],
                }
                Rows=pd.concat((Rows,pd.DataFrame(new_row)))
        

        if saveresults:
            saveresults_pickle(Rows, inputfile=file_input, outputfile=file_output, backup=True)



##############################
 EEG LDA


 10%|█         | 2/20 [00:13<02:01,  6.73s/it]


KeyboardInterrupt: 

In [ ]:
#Integrate EEG EMG
import mne
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import date
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import StratifiedKFold
import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

seed = 224

# keys of the performance dictionary
saveresults = True
filepath = "../Results/BCI_Performances/BCI_Performances_on_single_region.pkl"
file_input = filepath
file_output = filepath
miff = [8,12]
maff = [12,30]
freqs_band = tuple([(i,j) for i,j in zip(miff,maff)])
Features_name = "PowerSpectra - MeanMax band"
Comments= "-" # "-"
Region_Selection= "-" # "-"
PFR = False

n_folds=5
kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
today = date.today()
sfreq = 250

tutte_roi = 68
verbose = False

for DataType in ["EEG concat MEG"]:
    for Classifier_name, meth in zip(["LDA", "SVC", "RandomForest"],[LDA, SVC, RandomForestClassifier]):

        Rows = pd.DataFrame({
            "ROIs": [],
            "NROIs":[],
            "DataType": [],
            "freqs_band": [],  
            "subject": [],
            "Classifier": [],
            "Accuracy": [],
            "Features": [],
            "Comments": [],
            "Region Selection": [],
            "Date": []
        })
        print("\n\n"+"#"*30+"\n",DataType,Classifier_name)
        for subject_index in tqdm.tqdm(range(20)):

            data_2_MI_EEG = read_subject(data_folder, "EEG", "MI",subject_index, verbose=False)
            data_2_Rest_EEG =read_subject(data_folder, "EEG", "Baseline",subject_index, verbose=False)
            data_2_MI_MEG = read_subject(data_folder, "MEG", "MI",subject_index, verbose=False)
            data_2_Rest_MEG =read_subject(data_folder, "MEG", "Baseline",subject_index, verbose=False)

            WMI_EEG, WRe_EEG, FreqMI_EEG = compute_welch(data_2_MI_EEG, data_2_Rest_EEG, sfreq,
                                                            kargs_welch={"fmin":miff[0], "fmax":maff[-1]})
            # WMI.shape (trials, ROIs, freqs)
            labels_EEG = [1 for __ in range(len(WMI_EEG))] + [0 for __ in range(len(WRe_EEG))] 
            Feat_extraction_EEG = extract_features_from_spectra_more_bands(WMI_EEG, WRe_EEG, [ _ for _ in range(tutte_roi)], FreqMI_EEG,
                    min_freq_frature=miff, max_freq_frature=maff, 
                    select_mean=True, select_max=True,
                    reshape_features=False)

            Feat_MI_EEG, Feat_Rest_EEG, labls_MI_EEG, labls_Rest_EEG, all_regions_EEG = Feat_extraction_EEG
            # Feat_MI.shape (Trials, selected_ROIs, features for Roi)
            Feat_All_EEG = np.concatenate((Feat_MI_EEG, Feat_Rest_EEG), axis=0)
            labels_All_EEG= labels_EEG

            #################################################################################

            WMI_MEG, WRe_MEG, FreqMI_MEG = compute_welch(data_2_MI_MEG, data_2_Rest_MEG, sfreq,
                                                            kargs_welch={"fmin":miff[0], "fmax":maff[-1]})
            labels_MEG = [1 for __ in range(len(WMI_MEG))] + [0 for __ in range(len(WRe_MEG))] 
            Feat_extraction_MEG = extract_features_from_spectra_more_bands(WMI_MEG, WRe_MEG, [ _ for _ in range(tutte_roi)], FreqMI_MEG,
                    min_freq_frature=miff, max_freq_frature=maff, 
                    select_mean=True, select_max=True,
                    reshape_features=False)
            Feat_MI_MEG, Feat_Rest_MEG, labls_MI_MEG, labls_Rest_MEG, all_regions_MEG = Feat_extraction_MEG
            Feat_All_MEG = np.concatenate((Feat_MI_MEG, Feat_Rest_MEG), axis=0)
            labels_All_MEG= np.array(labels_MEG)


            for region_index in range(tutte_roi):

                fold_accuracies = []
                regions_selected_EEG = [region_index]
                regions_selected_MEG = [region_index]
                for fold_idx, (train_idx, val_idx) in enumerate(kf.split([0 for _ in range(len(labels_MEG))], labels_MEG)):
                    
                    Feat_All_regions_selected_EEG = Feat_All_EEG[:,regions_selected_EEG,:].reshape(Feat_All_EEG.shape[0],-1)
                    Feat_All_regions_selected_MEG = Feat_All_MEG[:,regions_selected_MEG,:].reshape(Feat_All_MEG.shape[0],-1)
                    # Feat_All_regions_selected_EEG.shape is (trials,features)
                    Feat_All_regions_selected_concat = np.concatenate((Feat_All_regions_selected_EEG,Feat_All_regions_selected_MEG),axis=-1)
                    #####################################################################################################################
                    acc, cm, model = train_single_model(Feat_All_regions_selected_concat, labels_All_MEG,
                                                        train_idx, val_idx, method=meth, seed=224)

                    fold_accuracies.append(acc)
                    if verbose:
                        print(f"Fold {fold_idx+1} accuracy: {acc:.4f}, Confusion matrix:")
                        print(cm)

                    #####################################################################################################################

                # Summary
                if verbose:
                    print("\n==================================")
                    print("Mean accuracy:", np.mean(fold_accuracies))
                    print("Std accuracy:", np.std(fold_accuracies))



                new_row = {
                    "ROIs": [[regions_selected_EEG,regions_selected_MEG]],
                    "NROIs":[len(regions_selected_EEG)],
                    "DataType": [DataType],
                    "freqs_band": [freqs_band],  
                    "subject": [subject_index],
                    "Classifier": [Classifier_name], #SVC, #RandomForest, #LDA
                    "Accuracy": [fold_accuracies],
                    "Features": [Features_name],
                    "Comments": [Comments],
                    "Region Selection": [Region_Selection],
                    "Date": [today],
                    "Personalized_Freq_Range": [PFR],
                }
                Rows=pd.concat((Rows,pd.DataFrame(new_row)))
        


        if saveresults:
            saveresults_pickle(Rows, inputfile=file_input, outputfile=file_output, backup=True)



##############################
 EEG concat MEG LDA


100%|██████████| 20/20 [01:00<00:00,  3.01s/it]


There are 8040 duplicated items (based on all columns except Date, ROIs, Performance).
 Index of the duplicated items: [   1    2    3 ... 8157 8158 8159]


##############################
 EEG concat MEG SVC


100%|██████████| 20/20 [00:59<00:00,  2.96s/it]


There are 9380 duplicated items (based on all columns except Date, ROIs, Performance).
 Index of the duplicated items: [   1    2    3 ... 9517 9518 9519]


##############################
 EEG concat MEG RandomForest


100%|██████████| 20/20 [05:03<00:00, 15.18s/it]

There are 10720 duplicated items (based on all columns except Date, ROIs, Performance).
 Index of the duplicated items: [    1     2     3 ... 10877 10878 10879]


# Test

In [ ]:
import pandas as pd
Rows = pd.DataFrame({
            "ROIs": [],
            "NROIs":[],
            "DataType": [],
            "freqs_band": [],  
            "subject": [],
            "Classifier": [],
            "Accuracy": [],
            "Features": [],
            "Comments": [],
            "Region Selection": [],
            "Date": []
        })
# Rows.to_pickle("../Results/BCI_Performances/BCI_Performances_region_selection_on_folds_8_30_Hz_features_on_3_bands.pkl")

In [ ]:
pd.read_pickle("../Results/BCI_Performances/BCI_Performances_region_selection_on_folds_8_30_Hz.pkl")

,ROIs,NROIs,DataType,freqs_band,subject,Classifier,Performance,Features,Comments,Date,Personalized_Freq_Range
0,"[23, 15, 14]",3.0,EEG,"((8, 12), (12, 30))",0.0,SVC,"[0.875, 0.8709677419354839, 0.9032258064516129...",PowerSpectra - MeanMax band,MyCriteria selection,2025-12-15,False
1,"[23, 15, 14, 22, 17]",5.0,EEG,"((8, 12), (12, 30))",0.0,SVC,"[0.9375, 0.8387096774193549, 0.838709677419354...",PowerSpectra - MeanMax band,MyCriteria selection,2025-12-15,False
2,"[23, 15, 14, 22, 17, 35, 42, 50, 6, 58]",10.0,EEG,"((8, 12), (12, 30))",0.0,SVC,"[0.9375, 0.8709677419354839, 0.870967741935483...",PowerSpectra - MeanMax band,MyCriteria selection,2025-12-15,False
3,"[23, 15, 14, 22, 17, 35, 42, 50, 6, 58, 51, 27...",20.0,EEG,"((8, 12), (12, 30))",0.0,SVC,"[0.96875, 0.8709677419354839, 0.90322580645161...",PowerSpectra - MeanMax band,MyCriteria selection,2025-12-15,False
4,"[67, 61, 43]",3.0,EEG,"((8, 12), (12, 30))",1.0,SVC,"[0.7368421052631579, 0.7368421052631579, 0.710...",PowerSpectra - MeanMax band,MyCriteria selection,2025-12-15,False
...,...,...,...,...,...,...,...,...,...,...,...
1435,"[[60, 62, 66, 44, 29, 18, 36, 25, 65, 39, 40, ...",20.0,EEG concat MEG,"((8, 12), (12, 30))",18.0,LDA,"[0.5428571428571428, 0.5714285714285714, 0.714...",PowerSpectra - MeanMax band,T test selection mean power alpha beta,2025-12-15,False
1436,"[[51, 21, 63], [44, 32, 47]]",3.0,EEG concat MEG,"((8, 12), (12, 30))",19.0,LDA,"[0.7368421052631579, 0.7105263157894737, 0.868...",PowerSpectra - MeanMax band,T test selection mean power alpha beta,2025-12-15,False
1437,"[[51, 21, 63, 33, 45], [44, 32, 47, 33, 46]]",5.0,EEG concat MEG,"((8, 12), (12, 30))",19.0,LDA,"[0.7894736842105263, 0.7631578947368421, 0.842...",PowerSpectra - MeanMax band,T test selection mean power alpha beta,2025-12-15,False
1438,"[[51, 21, 63, 33, 45, 55, 17, 35, 11, 49], [44...",10.0,EEG concat MEG,"((8, 12), (12, 30))",19.0,LDA,"[0.7368421052631579, 0.6842105263157895, 0.789...",PowerSpectra - MeanMax band,T test selection mean power alpha beta,2025-12-15,False


In [9]:
1440/80

18.0